# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Identify content ranking on page one of search results (average position between 1.0 and 10.0) that receives high organic visibility (90-day impressions at or above the 60th percentile), but whose click-through rate (CTR) is strictly below the cohort median for its position bucket.

We prioritize these URLs by the estimated volume of missed clicks:
missed_clicks = ((expected_ctr - ctr) / 100) * impressions_90d

These pages are already trusted by search engines to rank on page one, but searchers choose other listings or SERP features. Instead of expensive content overhauls to chase higher rank, these pages require targeted title tag and metadata snippet optimization.

Action Label & Reason Code

Action Label: OPTIMIZE_METADATA_SNIPPET

Reason Code: HIGH_IMP_LOW_CTR_PAGE1

Reason Code Trigger:

HIGH_IMP_LOW_CTR_PAGE1: Fired when avg_position is between 1.0 and 10.0, impressions_90d exceeds the 60th percentile, and ctr falls below the position cohort's median benchmark.

In [4]:
!git clone https://github.com/Omahcy/FlyRank_Internship.git
%cd /content/FlyRank_Internship

Cloning into 'FlyRank_Internship'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 166 (delta 71), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 1.85 MiB | 5.10 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/FlyRank_Internship


In [5]:
import pandas as pd
import numpy as np

# Load the historical dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total dataset size: {len(df)} rows\n")

# ==========================================
# SIGNAL 1: CTR vs Position Bucket (Flag-linked from session)
# Intuition: Higher positions (1-3) should yield significantly higher CTR,
# decaying predictably as rank drops.
# ==========================================
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, 100],
    labels=["1-3", "4-5", "6-10", "11-20", "21-50", "51-100"],
)

signal_1_bucket = (
    df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
    )
    .reset_index()
)

print("--- SIGNAL 1: Position vs CTR Bucket Table ---")
print(signal_1_bucket.to_string(index=False))
print("\nSignal 1 Verdict: CONFIRMED")
print(
    "Reasoning: Top ranks (1-3) exhibit higher median CTR, decaying steadily"
    " across lower ranks."
)
print(
    "Pages in positions 1-10 falling below their cohort median represent clear"
    " underperformers.\n"
)

# ==========================================
# SIGNAL 2: Impression Volume vs Click Yield
# Intuition: Higher 90-day impressions must reliably yield more clicks,
# proving that impression scale is a valid multiplier for prioritization.
# ==========================================
df["impression_bucket"] = pd.qcut(
    df["impressions_90d"].rank(method="first"),
    q=5,
    labels=["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"],
)

signal_2_bucket = (
    df.groupby("impression_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        mean_impressions=("impressions_90d", "mean"),
        mean_clicks=("clicks_90d", "mean"),
        median_clicks=("clicks_90d", "median"),
    )
    .reset_index()
)

print("--- SIGNAL 2: Impression Volume vs Click Yield ---")
print(signal_2_bucket.to_string(index=False))
print("\nSignal 2 Verdict: CONFIRMED")
print(
    "Reasoning: Mean and median clicks scale monotonically with impression"
    " quintiles."
)

Total dataset size: 30000 rows

--- SIGNAL 1: Position vs CTR Bucket Table ---
position_bucket    n  mean_ctr  median_ctr
            1-3 1141  2.714303        0.00
            4-5 2782  1.104820        0.23
           6-10 9060  0.511708        0.14
          11-20 7273  0.323443        0.10
          21-50 7225  0.222345        0.03
         51-100 1299  0.152525        0.00

Signal 1 Verdict: CONFIRMED
Reasoning: Top ranks (1-3) exhibit higher median CTR, decaying steadily across lower ranks.
Pages in positions 1-10 falling below their cohort median represent clear underperformers.

--- SIGNAL 2: Impression Volume vs Click Yield ---
impression_bucket    n  mean_impressions  mean_clicks  median_clicks
      Q1 (Lowest) 6000         10.119333     0.107333            0.0
               Q2 6000        165.549167     0.437833            0.0
               Q3 6000        775.670500     1.511500            1.0
               Q4 6000       2808.078500     7.527333            4.0
     Q5 (Hi

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 2. Define position cohorts and establish baseline benchmarks
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, 100],
    labels=["1-3", "4-5", "6-10", "11-20", "21-50", "51-100"],
)

# For 1-3, where median is 0.00 due to skew, use mean to avoid zeroing out candidates
cohort_stats = (
    df.groupby("position_bucket", observed=False)
    .agg(median_ctr=("ctr", "median"), mean_ctr=("ctr", "mean"))
    .reset_index()
)

cohort_stats["expected_ctr"] = np.where(
    cohort_stats["median_ctr"] > 0,
    cohort_stats["median_ctr"],
    cohort_stats["mean_ctr"],
)

df = df.merge(
    cohort_stats[["position_bucket", "expected_ctr"]],
    on="position_bucket",
    how="left",
)

# 3. Baseline rule: Page one (1-10), high volume (>= 60th percentile), CTR below expected
volume_threshold = df["impressions_90d"].quantile(0.60)
mask_rule = (
    (df["avg_position"] >= 1.0)
    & (df["avg_position"] <= 10.0)
    & (df["impressions_90d"] >= volume_threshold)
    & (df["ctr"] < df["expected_ctr"])
)

candidates = df[mask_rule].copy()

# 4. Score calculation: Estimated missed clicks scaled 0 to 100
candidates["missed_clicks"] = (
    (candidates["expected_ctr"] - candidates["ctr"]) / 100.0
) * candidates["impressions_90d"]

max_missed = candidates["missed_clicks"].max()
candidates["baseline_score"] = (
    (candidates["missed_clicks"] / max_missed) * 100.0
).round(2)

# Assign action label and reason code
candidates["reason_code"] = "HIGH_IMP_LOW_CTR_PAGE1"
candidates["action_label"] = "OPTIMIZE_METADATA_SNIPPET"

# 5. Build ranked queue
output_cols = [
    "content_id",
    "avg_position",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "expected_ctr",
    "baseline_score",
    "reason_code",
    "action_label",
]

ranked_queue = (
    candidates[output_cols]
    .sort_values(by="baseline_score", ascending=False)
    .reset_index(drop=True)
)

# 6. Write output CSV to work/outputs/baseline_action_score.csv
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "baseline_action_score.csv"
ranked_queue.to_csv(output_file, index=False)

print(f"Generated {output_file} with {len(ranked_queue)} candidate rows.\n")

Generated work/outputs/baseline_action_score.csv with 2070 candidate rows.



In [7]:
for idx, row in ranked_queue.head(20).iterrows():
    print(
        f"{idx}: {row['content_id']} | Pos: {row['avg_position']} | Imp:"
        f" {row['impressions_90d']} | CTR: {row['ctr']}% | Exp:"
        f" {row['expected_ctr']:.2f}% | Score: {row['baseline_score']}"
    )

0: content_8c19996aa890 | Pos: 2.5 | Imp: 509252 | CTR: 0.15% | Exp: 2.71% | Score: 100.0
1: content_4c36c775b818 | Pos: 2.3 | Imp: 463103 | CTR: 0.41% | Exp: 2.71% | Score: 81.72
2: content_8451fc6f034d | Pos: 2.3 | Imp: 272144 | CTR: 0.03% | Exp: 2.71% | Score: 55.94
3: content_44e481c8f55b | Pos: 1.4 | Imp: 312694 | CTR: 0.65% | Exp: 2.71% | Score: 49.43
4: content_9532f197bbc8 | Pos: 2.0 | Imp: 309192 | CTR: 0.87% | Exp: 2.71% | Score: 43.67
5: content_e12868d1f396 | Pos: 2.9 | Imp: 149712 | CTR: 0.07% | Exp: 2.71% | Score: 30.32
6: content_4a6607efcb46 | Pos: 2.2 | Imp: 128068 | CTR: 0.01% | Exp: 2.71% | Score: 26.52
7: content_4fc39a2b8cf0 | Pos: 2.6 | Imp: 160959 | CTR: 0.69% | Exp: 2.71% | Score: 24.95
8: content_11900bd7941a | Pos: 2.8 | Imp: 123561 | CTR: 0.41% | Exp: 2.71% | Score: 21.8
9: content_03d2673b2553 | Pos: 1.9 | Imp: 143314 | CTR: 0.83% | Exp: 2.71% | Score: 20.68
10: content_e5ae436f9a16 | Pos: 3.0 | Imp: 117741 | CTR: 0.45% | Exp: 2.71% | Score: 20.42
11: conten

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 Priority Review and Skepticism Checks**

1. **content_8c19996aa890**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.5, 509k impressions, 0.15% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Dominated by a massive zero-click SERP feature, like a detailed Google AI Overview, calculation tool, or expanded Knowledge Panel that resolves the query immediately on page.


2. **content_4c36c775b818**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.3, 463k impressions, 0.41% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Navigational search intent where users search for a specific competitor brand; position 1 captures 95%+ of all navigational clicks regardless of snippet quality.


3. **content_8451fc6f034d**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.3, 272k impressions, 0.03% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Top Sponsored Google Ads (3 to 4 ads) and rich product shopping feeds sit above organic position 2, visually pushing it down and capturing almost all transactional clicks.


4. **content_44e481c8f55b**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 1.4, 312k impressions, 0.65% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Featured Snippet cannibalization; the page itself or a direct competitor owns the position 0 answer box, meaning organic link clicks are inherently suppressed.


5. **content_9532f197bbc8**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.0, 309k impressions, 0.87% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Intent mismatch; the page ranks for a generic definition query while the content itself is a niche technical documentation piece.


6. **content_e12868d1f396**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.9, 149k impressions, 0.07% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Video or image carousel blocks appear prominently between positions 1 and 3, siphoning away standard text link interest.


7. **content_4a6607efcb46**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.2, 128k impressions, 0.01% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Ambiguous query where searchers are expecting a local physical business or government entity rather than an online article.


8. **content_4fc39a2b8cf0**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.6, 160k impressions, 0.69% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Competitor pages above and below utilize rich snippet schema (such as star ratings, pricing, and FAQ dropdowns), making this listing look less engaging.


9. **content_11900bd7941a**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.8, 123k impressions, 0.41% CTR vs 2.71% expected)
* **What Would Make It Wrong:** The snippet was recently modified in the CMS, but search engines are serving a stale cached snippet from prior to the update.


10. **content_03d2673b2553**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 1.9, 143k impressions, 0.83% CTR vs 2.71% expected)
* **What Would Make It Wrong:** The search volume is inflated by scraping bots or automated query tools that do not generate organic click interactions.


11. **content_e5ae436f9a16**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 3.0, 117k impressions, 0.45% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Position 3 sits just above the fold on mobile screens, but interactive SERP modules (such as People Also Ask) push it down for mobile searchers.


12. **content_654d006adc44**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.4, 131k impressions, 0.70% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Highly recognized category authority at position 1 (e.g., Wikipedia, CDC, or an official registry) dominates user trust.


13. **content_a19c8ee994c2**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 3.0, 124k impressions, 0.63% CTR vs 2.71% expected)
* **What Would Make It Wrong:** The meta title has already been hyper-optimized with click-inducing keywords; low CTR could stem from domain trust or brand reputation issues rather than phrasing.


14. **content_45636736e912**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 3.0, 112k impressions, 0.50% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Seasonal demand or temporal spike where searchers only browse for breaking news updates and ignore evergreen articles.


15. **content_23d452af4198**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.4, 117k impressions, 0.77% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Canonical/duplicate content issues where a sibling URL under the same domain also appears on page one and takes the click.


16. **content_4d1fe5b32dc2**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.5, 97k impressions, 0.52% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Searchers seek a downloadable tool, template, or calculator, while this URL title clearly denotes a purely descriptive informational post.


17. **content_07f2e7a6f38a**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.7, 101k impressions, 0.85% CTR vs 2.71% expected)
* **What Would Make It Wrong:** High geographic mismatch; the listing ranks nationally but searchers require region-specific service providers.


18. **content_6f81ccd92b64**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.9, 73k impressions, 0.19% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Rank instability; the URL experiences frequent ranking fluctuations between page 1 and page 2, giving misleading aggregate impression data.


19. **content_7df0362cda2d**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.8, 81k impressions, 0.76% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Search engine snippet generation overrides the provided meta tag with arbitrary on-page text, nullifying metadata optimization efforts.


20. **content_9c195417f6ef**
* **Action:** OPTIMIZE_METADATA_SNIPPET
* **Reason Code:** HIGH_IMP_LOW_CTR_PAGE1
* **Confidence Note:** High (Pos 2.5, 79k impressions, 0.73% CTR vs 2.71% expected)
* **What Would Make It Wrong:** Multi-intent search term where searchers predominantly look for physical product purchasing rather than the page's educational review format.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weak Picks Analysis

Zero-Click SERP Feature Traps (e.g., content_8c19996aa890 at Rank 2.5 with 509k impressions):

The rule heavily rewards raw impression scale. For top-tier ranks (1 to 3), low CTR frequently occurs when Google answers the query outright via an expanded AI Overview, interactive calculator, or Featured Snippet. Rewriting title tags or meta snippets will not recover missed clicks if searchers never intended to click an organic link in the first place.

Dominant Brand and Navigational Distortions (e.g., content_4c36c775b818 at Rank 2.3 with 463k impressions):

When a query is navigational, the brand owner at position 1 captures virtually all click volume. Second-position listings will exhibit low CTR regardless of copywriting quality. Flagging these URLs as optimization priorities overestimates recoverable traffic because user intent is committed to a different entity.

SERP Real Estate Compression (e.g., content_8451fc6f034d at Rank 2.3 with 0.03% CTR):

On commercially dense queries, top ad placements, Google Shopping carousels, and local packs compress organic rankings down the screen. An average position of 2.3 may actually sit below the visible fold on mobile screens, making snippet rewriting alone insufficient without broader SERP-layout context.

Leakage Verification Checklist

No Future Windows: The scoring and filtering logic uses exclusively historical 90-day aggregate metrics (avg_position, impressions_90d, clicks_90d, ctr). No 30-day post-intervention data (impressions_last_30d, clicks_last_30d, sessions_last_30d, trend_direction, trend_pct) was accessed or included.

No Label or Target Leakage: The cohort benchmark (expected_ctr) was calculated purely across baseline position cohorts from historical observations. No human refresh flags, evaluation tags, or post-hoc performance labels informed the scoring.

Pre-Intervention Ground Truth: All candidate selection rules and score calculations rely strictly on metrics observable at decision time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.